# 00. Agentic AI 실습 환경 점검

- 선수 실습: 없음
- 실습 난이도: 시작하기
- 최종 산출물: Python·패키지·경로·API·Chroma 점검 결과

이 Notebook을 가장 먼저 실행한다. 모든 항목이 통과하면 01-1번부터 순서대로
진행한다.


## 1. 학습목표

1. 공통 `agentic_ai` 패키지가 올바르게 설치되었는지 확인할 수 있다.
2. `.env`, 데이터, 출력 경로를 확인할 수 있다.
3. Chat·Embedding 모델과 Chroma가 실제로 동작하는지 검증할 수 있다.
4. 실패 항목에 따라 어떤 설정을 수정해야 하는지 판단할 수 있다.


## 2. 실행 전 준비

저장소 루트(`agentic_ai_lab`)에서 다음 명령을 한 번 실행한다.  
이 프로젝트는 루트의 `pyproject.toml`, `uv.lock`, `.venv`를 공통으로 사용한다.  

```bash
uv sync
```

`.env.example`을 `.env`로 복사하고 `OPENAI_API_KEY`를 입력한다.  
API Key 값은 Notebook에 직접 작성하거나 출력하지 않는다.  


## 3. Python과 공통 패키지 확인

첫 번째 셀은 `src/agentic_ai`가 설치되어 있는지 확인한다.  
import에 실패하면 `uv sync`를 다시 실행하고 루트 `.venv` Kernel을 선택한 뒤 재시작한다.  


In [11]:
import sys

try:
    import agentic_ai
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "agentic_ai 패키지를 찾지 못했습니다. "
        "저장소 루트에서 'uv sync'를 실행하고 루트 .venv를 "
        "Jupyter Kernel로 선택한 뒤 재시작하세요."
    ) from exc

In [12]:
from agentic_ai.config import get_settings
from agentic_ai.notebook_utils import environment_report, print_environment_summary
from agentic_ai.paths import DATA_DIR, OUTPUT_DIR, PROJECT_ROOT

In [13]:
settings = get_settings()
print_environment_summary(
    settings,
    needs_chat_model=True,
    needs_embedding_model=True,
)

report = environment_report()
assert sys.version_info >= (3, 12), "Python 3.12 이상이 필요합니다."
assert report["data_dir_exists"], "data 폴더를 찾지 못했습니다."
assert all(version != "설치되지 않음" for version in report["packages"].values())

print("\n[핵심 패키지]")
for package, version in report["packages"].items():
    print(f"- {package}: {version}")


[환경 설정 확인]
- 프로젝트: D:\frodo\agentic_ai_lab
- 데이터: D:\frodo\agentic_ai_lab\data
- 출력: D:\frodo\agentic_ai_lab\outputs
- OPENAI_API_KEY: 설정됨
- Chat Model: gpt-5-mini
- Embedding Model: text-embedding-3-small

[핵심 패키지]
- langchain: 1.3.14
- langgraph: 1.2.9
- langchain-openai: 1.3.5
- langchain-chroma: 1.1.0
- pydantic: 2.13.4


**결과 해석**: 프로젝트·데이터·출력 경로가 모두 `agentic_ai_lab` 아래를
가리키고 핵심 패키지에 버전이 표시되면 기본 설치가 정상이다.


## 4. 출력 폴더 쓰기 확인

임시 파일을 만들고 읽은 뒤 바로 삭제한다. 이 검사는 실습 로그를 저장할 권한이
있는지 확인한다.


In [14]:
check_file = OUTPUT_DIR / "_environment_write_check.txt"
check_file.write_text("environment-ok", encoding="utf-8")
assert check_file.read_text(encoding="utf-8") == "environment-ok"
check_file.unlink()
print("출력 폴더 쓰기 확인 완료")


출력 폴더 쓰기 확인 완료


## 5. Chat Model 연결 확인

이 셀부터 실제 OpenAI API를 호출한다. 실패하면 `.env`의 API Key와 Chat Model명을
확인한다.


In [15]:
from agentic_ai.models import get_chat_model

assert settings.api_key_configured, "OPENAI_API_KEY가 설정되지 않았습니다."
chat_model = get_chat_model()
chat_response = chat_model.invoke("환경 점검입니다. READY 한 단어로만 답하세요.")
assert chat_response.content
print("Chat Model 응답:", chat_response.content)


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: your_api*************************************************************************************************************************************************************************LskA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

## 6. Embedding Model 연결 확인

짧은 문장을 Embedding하여 숫자 벡터가 반환되는지 확인한다.


In [ ]:
from agentic_ai.models import get_embedding_model

embedding_model = get_embedding_model()
test_vector = embedding_model.embed_query("Agentic AI 환경 점검")

assert len(test_vector) > 0
assert all(isinstance(value, float) for value in test_vector[:5])
print(f"Embedding 차원: {len(test_vector)}")
print("앞의 3개 값:", test_vector[:3])


Embedding 차원: 1536
앞의 3개 값: [-0.0261993408203125, 0.05340576171875, -0.002002716064453125]


## 7. Chroma 저장·검색 확인

환경 점검 전용 Collection에 문서 하나를 저장하고 검색한다. 실습 Collection과
이름이 다르므로 이후 실습 데이터에는 영향을 주지 않는다.


In [ ]:
from langchain_core.documents import Document

from agentic_ai.retrieval_utils import add_documents_if_empty, get_chroma_store

check_store = get_chroma_store(
    "environment_check",
    embedding_model=embedding_model,
)

added, count = add_documents_if_empty(
    check_store,
    [Document(page_content="Agentic AI 환경 점검 문서", metadata={"type": "check"})],
    ids=["environment-check-1"],
)
matches = check_store.similarity_search("환경 점검", k=1)

assert matches and matches[0].metadata["type"] == "check"
print(f"Chroma 저장·검색 확인 완료 (문서 수: {count}, 새로 추가: {added})")

# 점검 전용 Collection만 정리한다.
check_store.delete_collection()


Chroma 저장·검색 확인 완료 (문서 수: 1, 새로 추가: True)


## 8. 최종 점검 결과


In [ ]:
final_checks = {
    "Python": True,
    "공통 패키지": True,
    "경로와 출력 권한": True,
    "Chat Model": bool(chat_response.content),
    "Embedding Model": len(test_vector) > 0,
    "Chroma": bool(matches),
}

for name, passed in final_checks.items():
    print(f"{'PASS' if passed else 'FAIL':4} | {name}")

# assert 조건이 False이면 AssertionError를 발생시키고, 모든 점검이 통과했음을 알리는 메시지를 출력한다.
assert all(final_checks.values())
print("\n모든 환경 점검을 통과했습니다. 01-1번 실습을 시작하세요.")


PASS | Python
PASS | 공통 패키지
PASS | 경로와 출력 권한
PASS | Chat Model
PASS | Embedding Model
PASS | Chroma

모든 환경 점검을 통과했습니다. 01-1번 실습을 시작하세요.


## 9. 오류가 발생할 때

| 증상 | 확인할 내용 |
|---|---|
| `No module named agentic_ai` | 저장소 루트에서 `uv sync` 후 루트 `.venv` Kernel 선택·재시작 |
| 특정 패키지 Import 오류 | 저장소 루트에서 `uv add <패키지명>` 후 Kernel 재시작 |
| API Key 오류 | `.env` 위치와 `OPENAI_API_KEY` 값 |
| Model not found | `.env`의 `CHAT_MODEL`, `EMBEDDING_MODEL` |
| 데이터 경로 오류 | Notebook을 프로젝트의 Jupyter 환경에서 열었는지 확인 |
| Chroma 오류 | `langchain-chroma` 설치와 출력 폴더 권한 확인 |

API Key나 `.env` 내용 전체를 화면에 출력하지 않는다.
